# Kafka Demo — Lab 3

### Connect to Kafka Broker Server
Open an SSH tunnel in your terminal and leave it running while you use this notebook.
Replace `<NetID>` with your UIC NetID:
```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

### To kill connection
```
lsof -ti:9092 | xargs kill -9
```

### Setup
```
python -m pip install kafka-python
```

See [bug_list.md](./bug_list.md) for frequent bugs and solutions.

In [1]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer

# Update this for your own recitation section :)
topic = 'recitation-x' # replace x with your recitation section

### Producer Mode -> Writes Data to Broker

In [2]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# [TODO]: Replace '...' with the address of your Kafka bootstrap server
producer = KafkaProducer(bootstrap_servers=['localhost:9092'],
                        value_serializer=lambda x: dumps(x).encode('utf-8'))

# [TODO]: Add cities of your choice
cities = ['Chicago', 'Jacksonville', 'Bloomington']

# Write data via the producer
print("Writing to Kafka Broker")
for i in range(10):
    data = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")},{cities[randint(0,len(cities)-1)]},{randint(18, 32)}ºC'
    print(f"Writing: {data}")
    producer.send(topic=topic, value=data)
    sleep(1)

Writing to Kafka Broker
Writing: 2026-09-24 20:27:11,Jacksonville,28ºC
Writing: 2026-09-24 20:27:12,Jacksonville,23ºC
Writing: 2026-09-24 20:27:13,Bloomington,23ºC
Writing: 2026-09-24 20:27:14,Chicago,20ºC
Writing: 2026-09-24 20:27:15,Bloomington,29ºC
Writing: 2026-09-24 20:27:16,Bloomington,27ºC
Writing: 2026-09-24 20:27:17,Bloomington,30ºC
Writing: 2026-09-24 20:27:18,Chicago,31ºC
Writing: 2026-09-24 20:27:19,Chicago,18ºC
Writing: 2026-09-24 20:27:20,Bloomington,23ºC


### Consumer Mode -> Reads Data from Broker

In [ ]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# [TODO]: Complete the missing ... parameters/arguments using the Kafka documentation
consumer = KafkaConsumer(
    topic,
    bootstrap_servers=['localhost:9092'],
    group_id='kpate649-lab3',
    auto_offset_reset='earliest', #Experiment with different values
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000
)

print('Reading Kafka Broker')
for message in consumer:
    message = message.value.decode()
    # Default message.value type is bytes!
    print(loads(message))
    os.system(f"echo {message} >> kafka_log.csv")

Reading Kafka Broker
2026-09-23 18:06:18,chicago,30ºC
2026-09-23 18:06:20,chicago,28ºC
2026-09-23 18:06:21,san diego,21ºC
2026-09-23 18:06:22,san diego,28ºC
2026-09-23 18:06:23,san diego,27ºC
2026-09-23 18:06:24,chicago,30ºC
2026-09-23 18:06:25,phoenix,30ºC
2026-09-23 18:06:26,san diego,29ºC
2026-09-23 18:06:27,chicago,21ºC
2026-09-23 18:06:28,san diego,30ºC
2026-09-23 18:29:45,phoenix,31ºC
2026-09-23 18:29:46,phoenix,21ºC
2026-09-23 18:29:47,san diego,26ºC
2026-09-23 18:29:48,chicago,22ºC
2026-09-23 18:29:49,phoenix,23ºC
2026-09-23 18:29:50,san diego,22ºC
2026-09-23 18:29:51,san diego,23ºC
2026-09-23 18:29:52,chicago,22ºC
2026-09-23 18:29:53,san diego,18ºC
2026-09-23 18:29:54,chicago,21ºC
2026-09-23 21:26:21,seattle,23ºC
2026-09-23 21:26:22,portland,21ºC
2026-09-23 21:26:23,chicago,29ºC
2026-09-23 21:26:24,chicago,29ºC
2026-09-23 21:26:25,chicago,23ºC
2026-09-23 21:26:26,seattle,21ºC
2026-09-23 21:26:27,portland,20ºC
2026-09-23 21:26:28,portland,24ºC
2026-09-23 21:26:29,chicago,21ºC
2

# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
#kcat command: connect to local Kafka broker, specify a topic, and consume messages from the earliest offset